# requires-grad-propagation — worked example 3: Trace requires_grad Through a Chain of Operations

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `requires-grad-propagation`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """A minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries an optional `.recipe`
    populated by wrap_forward_fn. `requires_grad` is set by the wrapper."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

When operations are chained, each output's `requires_grad` is determined by the three-gate rule applied at that step, using the previous step's output as input. If the first operation produces a tracked output, subsequent differentiable operations will also produce tracked outputs. A single `no_grad` step anywhere in the chain breaks all downstream tracking.

## Worked solution

**Step 1 — create a tracked leaf.** `x` with `requires_grad=True` is the starting point of the chain.

**Step 2 — apply a chain of ops.** Each operation takes the previous output as input. Use `t.exp`, `t.log`, and `t.sqrt` — all differentiable — to build a three-step chain.

**Step 3 — verify each output has grad.** Every intermediate output should have `requires_grad=True` because the tracked leaf propagates through differentiable ops.

**Step 4 — break the chain with `detach()`.** Insert `.detach()` in the middle, which acts like a gate that stops gradient propagation. Subsequent ops produce `requires_grad=False`.

**Step 5 — confirm with the rule.** The rule predicts: any input without `requires_grad` → output without `requires_grad` (even if op is differentiable).

In [ ]:
import torch as t

# --- exercise and print ---
t.manual_seed(1)
x = t.tensor([0.5, 1.0, 2.0], requires_grad=True)

# Unbroken chain
y1 = t.exp(x)
y2 = t.log(y1 + 1.0)
y3 = t.sqrt(y2 + 0.1)

print('Unbroken chain (all should be True):')
print(f'  x.requires_grad: {x.requires_grad}')
print(f'  y1.requires_grad: {y1.requires_grad}')
print(f'  y2.requires_grad: {y2.requires_grad}')
print(f'  y3.requires_grad: {y3.requires_grad}')

# Detach in the middle
z1 = t.exp(x)
z2 = z1.detach()        # grad stops here
z3 = t.log(z2 + 1.0)
z4 = t.sqrt(z3 + 0.1)

print('\nChain with detach in the middle:')
print(f'  z1.requires_grad: {z1.requires_grad}')  # True
print(f'  z2.requires_grad: {z2.requires_grad}')  # False (detached)
print(f'  z3.requires_grad: {z3.requires_grad}')  # False
print(f'  z4.requires_grad: {z4.requires_grad}')  # False

print('\nAll-True chain correct:', all([y1.requires_grad, y2.requires_grad, y3.requires_grad]))
print('Post-detach all False: ', not any([z2.requires_grad, z3.requires_grad, z4.requires_grad]))